# α-BaCu₂Se₂ synthesis parameter extraction

Pulls synthesis records from the NOMAD Oasis REST API (token in `.env`), restricted to `INLThinFilmStack` samples. Falls back to a local archive export under `.volumes/fs/staging/`, then to the processed synthesis CSV. Writes the complete database to `data/processed/bacuse2_synthesis_database_nomad.csv`, including selenization pulse fields (`se_pulse_on_ms`, `se_pulse_off_ms`, `se_pulse_count`).

In [3]:
import sys
from pathlib import Path

# Make the package importable when the kernel runs from the notebooks/ folder.
sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd

from xrd_bo_optimization_tf.config import (
    NOMAD_STAGING_DIR,
    NOMAD_SYNTHESIS_CSV,
    NOMAD_VOLUME_DIR,
    SYNTHESIS_CSV,
)
from xrd_bo_optimization_tf.nomad_client import iter_archive_files
from xrd_bo_optimization_tf.synthesis import (
    build_synthesis_dataframe,
    load_synthesis_archives,
    load_synthesis_from_nomad,
)

# Prefer the live NOMAD API. Fall back to a local archive export, then to the CSV.
try:
    synthesis_db = load_synthesis_from_nomad(NOMAD_STAGING_DIR)
    data_source = f"NOMAD API ({len(synthesis_db)} samples)"
except Exception as nomad_error:  # noqa: BLE001 - offline fallback
    print(f"NOMAD API unavailable ({nomad_error}); trying local archives.")
    archive_paths = list(iter_archive_files(NOMAD_VOLUME_DIR))
    synthesis_db = load_synthesis_archives(NOMAD_VOLUME_DIR) if archive_paths else {}
    data_source = f"{len(archive_paths)} local NOMAD archive files"

if synthesis_db:
    df = build_synthesis_dataframe(synthesis_db)
    NOMAD_SYNTHESIS_CSV.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(NOMAD_SYNTHESIS_CSV, index=False)
    print(f"Wrote complete synthesis table to {NOMAD_SYNTHESIS_CSV}")
elif SYNTHESIS_CSV.is_file():
    df = pd.read_csv(SYNTHESIS_CSV)
    data_source = str(SYNTHESIS_CSV)
else:
    raise FileNotFoundError(
        f"No NOMAD source reachable and no synthesis CSV at {SYNTHESIS_CSV}"
    )

print(f"Loaded {len(df)} synthesis records from {data_source}")
df.head()

/home/nomad-lanasc/xrd-bo-optimization-tf/xrd_bo_env/lib/python3.12/site-packages/urllib3/connectionpool.py:1129: InsecureRequestWarning: Unverified HTTPS request is being made to host 'nomad-sadewasser.inl.local'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
/home/nomad-lanasc/xrd-bo-optimization-tf/xrd_bo_env/lib/python3.12/site-packages/urllib3/connectionpool.py:1129: InsecureRequestWarning: Unverified HTTPS request is being made to host 'nomad-sadewasser.inl.local'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
/home/nomad-lanasc/xrd-bo-optimization-tf/xrd_bo_env/lib/python3.12/site-packages/urllib3/connectionpool.py:1129: InsecureRequestWarning: Unverified HTTPS request is being made to host 'nomad-sadewasser.inl.local'. Adding certificate verification is strongly advised. See:

Wrote complete synthesis table to /home/nomad-lanasc/xrd-bo-optimization-tf/data/processed/bacuse2_synthesis_database_nomad.csv
Loaded 42 synthesis records from NOMAD API (42 samples)


,sample_id,series,completeness,spin_speed_rpm,spin_duration_s,spin_acceleration,soft_bake_temp_c,soft_bake_duration_s,soft_bake_duration_min,hard_bake_temp_c,...,selenization_temp_c,selenization_time_s,selenization_pressure_pa,se_pulse_on_ms,se_pulse_off_ms,se_pulse_count,solution_used_name,solution_volume_ml,num_archives,has_annealing
0,260429A11,260429A,complete,2000.0,20.0,1000.0,100.0,120.0,2.0,250.0,...,520.0,600.0,0.578,500.0,2700.0,188,260427A1 0.1 milliliter,0.1,1,True
1,260429A12,260429A,complete,2000.0,20.0,1000.0,100.0,120.0,2.0,250.0,...,520.0,600.0,0.578,500.0,2700.0,188,260427A1 0.1 milliliter,0.1,1,True
2,260429A2,260429A,complete,2000.0,20.0,1000.0,100.0,120.0,2.0,250.0,...,540.0,1200.0,0.582,500.0,2700.0,375,260427A2 0.1 milliliter,0.1,1,True
3,260429A3,260429A,complete,2000.0,20.0,1000.0,100.0,120.0,2.0,250.0,...,560.0,1800.0,0.576,500.0,2700.0,562,260427A3 0.1 milliliter,0.1,1,True
4,260429B1,260429B,complete,2000.0,20.0,1000.0,100.0,120.0,2.0,250.0,...,520.0,1200.0,0.580,500.0,2700.0,375,260427B1 0.1 milliliter,0.1,1,True


In [4]:
parameter_columns = [
    'spin_speed_rpm',
    'soft_bake_temp_c',
    'hard_bake_temp_c',
    'selenization_temp_c',
    'selenization_time_s',
]
available_columns = [column for column in parameter_columns if column in df]
completeness = df[available_columns].notna().sum().rename('available_samples').to_frame()
completeness['total_samples'] = len(df)
completeness

,available_samples,total_samples
spin_speed_rpm,42,42
soft_bake_temp_c,42,42
hard_bake_temp_c,42,42
selenization_temp_c,35,42
selenization_time_s,37,42


In [5]:
from xrd_bo_optimization_tf.nomad_api import load_upload_ids

try:
    upload_ids = load_upload_ids()
    print(f'{len(upload_ids)} BaCuSe uploads configured:')
    for upload_id in upload_ids:
        print(f'  {upload_id}')
except Exception as error:  # noqa: BLE001
    print(f'Upload-ID list unavailable: {error}')

3 BaCuSe uploads configured:
  mZdfUGtwRr-qtYuvxb0lfQ
  TBRSU9POTmKRrd1aytChkA
  csHzBMzSTu-jLd38eB_POg
